# QA 02 — logistic regression

Binary and multiclass links, labels, thresholds, imbalance, preprocessing, and predictions.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
from mlektic import explain_logistic_prediction, visualize_logistic
from notebooks._support import case_heading, binary_case, multiclass_case, scaled_binary_case

### `LOG-BINARY-INDEXED`

**Inspect:** default class indices without semantic label noise

In [ ]:
case_heading("LOG-BINARY-INDEXED", "default class indices without semantic label noise")
c=binary_case(1,string_labels=True)
display(visualize_logistic(c.model,c.X,c.y,steps=12,max_frames=7,show_loss=True,show_class_labels=False))

### `LOG-BINARY-LABELS`

**Inspect:** opt-in semantic labels appended to class indices

In [ ]:
case_heading("LOG-BINARY-LABELS", "opt-in semantic labels appended to class indices")
c=binary_case(1,string_labels=True)
display(visualize_logistic(c.model,c.X,c.y,steps=12,max_frames=7,show_loss=True,show_class_labels=True))

### `LOG-BINARY-THRESHOLD`

**Inspect:** non-default binary threshold with numeric tick and separate τ label inside the geometry

In [ ]:
case_heading("LOG-BINARY-THRESHOLD", "non-default binary threshold with numeric tick and separate τ label inside the geometry")
c=binary_case(2)
display(visualize_logistic(c.model,c.X,c.y,steps=10,max_frames=6,threshold=0.7,detail='academic'))

### `LOG-BINARY-IMBALANCED`

**Inspect:** imbalanced data with empirical log-loss and metrics

In [ ]:
case_heading("LOG-BINARY-IMBALANCED", "imbalanced data with empirical log-loss and metrics")
c=binary_case(2,imbalanced=True)
display(visualize_logistic(c.model,c.X,c.y,steps=10,max_frames=6,show_loss=True,detail='complete'))

### `LOG-BINARY-30D-WRAP`

**Inspect:** thirty-variable equation defines sigmoid once, wraps the intercept, and centers metric readouts

In [ ]:
case_heading("LOG-BINARY-30D-WRAP", "thirty-variable equation defines sigmoid once, wraps the intercept, and centers metric readouts")
c=binary_case(30)
display(visualize_logistic(c.model,c.X,c.y,steps=12,max_frames=6,show_loss=True,metrics=['log_loss','accuracy','f1'],detail='complete',format='compact',size='wide'))

### `LOG-SGD-REPLAY`

**Inspect:** incremental classifier replay with EMA display smoothing

In [ ]:
case_heading("LOG-SGD-REPLAY", "incremental classifier replay with EMA display smoothing")
c=binary_case(2,estimator='sgd')
display(visualize_logistic(c.model,c.X,c.y,steps=20,max_frames=8,show_loss=True,smooth='ema'))

### `LOG-PIPE-ORIGINAL`

**Inspect:** scaled pipeline explained in original units

In [ ]:
case_heading("LOG-PIPE-ORIGINAL", "scaled pipeline explained in original units")
c=scaled_binary_case(2)
display(visualize_logistic(c.model,c.X,c.y,steps=10,max_frames=6,display_space='original',detail='complete'))

### `LOG-PIPE-SCALED`

**Inspect:** scaled pipeline coefficients in transformed units

In [ ]:
case_heading("LOG-PIPE-SCALED", "scaled pipeline coefficients in transformed units")
c=scaled_binary_case(2)
display(visualize_logistic(c.model,c.X,c.y,steps=10,max_frames=6,display_space='scaled',detail='complete'))

### `LOG-MULTI-FOCUS`

**Inspect:** one selected class surface while preserving fitted order

In [ ]:
case_heading("LOG-MULTI-FOCUS", "one selected class surface while preserving fitted order")
c=multiclass_case(2,string_labels=True)
display(visualize_logistic(c.model,c.X,c.y,steps=10,max_frames=6,class_focus='group-1',show_class_labels=True,show_loss=True,detail='academic'))

### `LOG-MULTI-OVR`

**Inspect:** normalized one-vs-rest semantics from SGDClassifier

In [ ]:
case_heading("LOG-MULTI-OVR", "normalized one-vs-rest semantics from SGDClassifier")
c=multiclass_case(2)
from sklearn.linear_model import SGDClassifier
m=SGDClassifier(loss='log_loss',max_iter=1000,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=12,max_frames=6,multiclass_link='ovr',show_loss=True,detail='academic'))

### `LOG-MULTI-MANY-CLASSES`

**Inspect:** six-class matrix truncation and ellipsis

In [ ]:
case_heading("LOG-MULTI-MANY-CLASSES", "six-class matrix truncation and ellipsis")
c=multiclass_case(6,classes=6,string_labels=True)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,max_theta_cols=3,detail='complete',size='wide'))

### `LOG-PRED-BINARY`

**Inspect:** winning class index, both probabilities, curve, and boxed result

In [ ]:
case_heading("LOG-PRED-BINARY", "winning class index, both probabilities, curve, and boxed result")
c=binary_case(1,string_labels=True)
display(explain_logistic_prediction(c.model,c.X,c.y,x_query=[[0.4]],show_class_labels=False))

### `LOG-PRED-LABELS`

**Inspect:** optional semantic winning label in prediction explanation

In [ ]:
case_heading("LOG-PRED-LABELS", "optional semantic winning label in prediction explanation")
c=binary_case(2,string_labels=True)
display(explain_logistic_prediction(c.model,c.X,c.y,x_query=c.X[[4]],show_class_labels=True,size='wide'))

### `LOG-PRED-MULTI`

**Inspect:** multiclass probability vector and argmax winner

In [ ]:
case_heading("LOG-PRED-MULTI", "multiclass probability vector and argmax winner")
c=multiclass_case(2,string_labels=True)
display(explain_logistic_prediction(c.model,c.X,c.y,x_query=c.X[[8]],show_class_labels=True))

## Reading the multiclass layouts

The animated coefficient matrices and equations display at most four decimal
places (`min(dec, 4)`); the fitted-model derivation below the figure honors
the requested `dec`. The short `log-loss` card uses the history source stated
in the subtitle and timeline; full loss provenance also remains in metadata.

`default`, `wide`, and `classroom` share the expanded multivariable notation:
endpoint scores on one line and numeric exponents in the probability fractions.
`compact` and `notebook` keep shorter symbolic fractions. Check the endpoint
continuation marks, the gap below the generic probability definition, and the
centering of the vertical ellipsis for both zero and fitted coefficients.
Omitted matrix columns are indicated by conventional ellipses, without an
extra caption. In binary 1D predictions, inspect the Substitution and Output
stages: the single-input panel gives the sigmoid derivation more vertical room.

## Canvas-size coverage

Run the same representative figure in every supported size. Inspect clipping,
mathematical spacing, controls, legends, axes, and any below-plot derivation.

### `LOG-ND-SIZE-DEFAULT`

**Inspect:** default canvas preset for this notebook's representative figure

In [ ]:
case_heading("LOG-ND-SIZE-DEFAULT", "default canvas preset for this notebook's representative figure")
c=multiclass_case(6,classes=6)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,show_loss=True,metrics=['log_loss','accuracy','f1'],max_theta_cols=2,detail='complete',size='default'))

### `LOG-ND-SIZE-COMPACT`

**Inspect:** compact canvas preset for this notebook's representative figure

In [ ]:
case_heading("LOG-ND-SIZE-COMPACT", "compact canvas preset for this notebook's representative figure")
c=multiclass_case(6,classes=6)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,show_loss=True,metrics=['log_loss','accuracy','f1'],max_theta_cols=2,detail='complete',size='compact'))

### `LOG-ND-SIZE-NOTEBOOK`

**Inspect:** notebook canvas preset for this notebook's representative figure

In [ ]:
case_heading("LOG-ND-SIZE-NOTEBOOK", "notebook canvas preset for this notebook's representative figure")
c=multiclass_case(6,classes=6)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,show_loss=True,metrics=['log_loss','accuracy','f1'],max_theta_cols=2,detail='complete',size='notebook'))

### `LOG-ND-SIZE-WIDE`

**Inspect:** wide canvas preset for this notebook's representative figure

In [ ]:
case_heading("LOG-ND-SIZE-WIDE", "wide canvas preset for this notebook's representative figure")
c=multiclass_case(6,classes=6)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,show_loss=True,metrics=['log_loss','accuracy','f1'],max_theta_cols=2,detail='complete',size='wide'))

### `LOG-ND-SIZE-CLASSROOM`

**Inspect:** classroom canvas preset for this notebook's representative figure

In [ ]:
case_heading("LOG-ND-SIZE-CLASSROOM", "classroom canvas preset for this notebook's representative figure")
c=multiclass_case(6,classes=6)
display(visualize_logistic(c.model,c.X,c.y,steps=8,max_frames=5,show_loss=True,metrics=['log_loss','accuracy','f1'],max_theta_cols=2,detail='complete',size='classroom'))